In [ ]:

import duckdb as dd
import pandas as pd
from embeddings import *
from duckdb_handler import *
from utils import *
from enrichment import *
from cleaningdata import *
import polars as pl
import json


#df = pd.read_csv(r"C:\Users\samar\ai_ml_data_pipeline\data\input_data\tech_news.csv")
#metadata = pd.read_json(r"C:\Users\samar\ai_ml_data_pipeline\data\input_data\company_metadata.json",  orient='index').reset_index()
#metadata.rename(columns={'index': 'company_name'}, inplace=True)

df = pl.read_csv(r"C:\Users\samar\ai_ml_data_pipeline\data\input_data\tech_news.csv")
df = df.to_pandas()

with open(r"C:\Users\samar\ai_ml_data_pipeline\data\input_data\company_metadata.json") as f:
    metadata =json.load(f)

metadata = pl.from_dicts(
    [{"company_name": k, **v} for k, v in metadata.items()]
)
metadata = metadata.to_pandas()

In [4]:
df["revenue_usd"] = df["revenue"].apply(parse_revenue)

df["published_date"] = df["published_date"].apply(normalize_date)

df = extract_date_parts(df)

df["category"] = df["category"].apply(normalize_category)

 # Enrichment (join)
df = df.merge(metadata, on="company_name", how="left")

df["company_age"] = df.apply(
    lambda x: compute_company_age(x["founded_year"], x["published_date"]), axis=1
)
df["company_size_category"] = df["employee_count"].apply(company_size_category)

# Embeddings
df = add_embeddings(df)

# Similarity
df = add_top_similar(df)

# Filter
df = filter_ai_articles(df)





c:\Users\samar\ai_ml_data_pipeline\etl\cleaningdata.py:56: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["quarter"] = df["published_date"].dt.to_period("Q")


In [5]:

print(df["published_date"].dtype)


datetime64[us, UTC]


In [6]:
con=load_to_duckdb(df)

query_result = con.sql("""
    SELECT *
    FROM articles
    WHERE revenue_usd >= 50000000
    AND year BETWEEN 2023 AND 2024
    AND (category = 'AI_ML' OR industry = 'AI/ML')
""")

# Fetch results as a Pandas DataFrame
print(query_result.df())
con.close()


   article_id                                              title company_name  \
0     ART0004            Confluent Reports Record Revenue Growth    Confluent   
1     ART0029  NVIDIA Announces Breakthrough in Large Languag...       NVIDIA   
2     ART0036             NVIDIA Launches New AI-Powered Product       NVIDIA   
3     ART0040                  SpaceX CEO Discusses Future of AI       SpaceX   
4     ART0079             Scale AI Raises Series D Funding Round     Scale AI   
5     ART0100          Cloudflare Releases Open Source Framework   Cloudflare   
6     ART0119   Cloudflare Partners with Major Enterprise Client   Cloudflare   
7     ART0121  SpaceX Announces Breakthrough in Large Languag...       SpaceX   
8     ART0219             Scale AI Raises Series D Funding Round     Scale AI   
9     ART0222                   SpaceX Faces Regulatory Scrutiny       SpaceX   
10    ART0223              NVIDIA Releases Open Source Framework       NVIDIA   
11    ART0249            Ant

In [34]:
df.dtypes


article_id                               str
title                                    str
company_name                             str
published_date           datetime64[us, UTC]
category                                 str
revenue                                  str
summary                                  str
url                                      str
author                                   str
word_count                           float64
revenue_usd                            int64
year                                   int32
month                                  int32
quarter                                  str
founded_year                           int64
headquarters                             str
employee_count                         int64
industry                                 str
is_public                               bool
stock_ticker                             str
company_age                            int64
company_size_category                    str
combined_t

In [29]:
df["category"] = df["category"].apply(normalize_category)

datetime64[us, UTC]
0    2020.0
1       NaN
2       NaN
Name: date, dtype: float64


In [41]:
import pandas as pd
import numpy as np

# Sample DataFrame with mixed date formats
df = pd.DataFrame({
    'Date_Column': ['1/05/2015', '15 Jul 2009', '1-Feb-15', '12/08/2019', 'not a date','2020-08-22T00:00:00Z', '2021-09-11T00:00:00Z']
})

# Convert the column to datetime using format='mixed' and errors='coerce'
df['Date_Column_Converted'] = pd.to_datetime(df['Date_Column'], format='mixed', utc=True, errors='coerce')

print(df[['Date_Column', 'Date_Column_Converted']])
print(df.dtypes)

            Date_Column     Date_Column_Converted
0             1/05/2015 2015-01-05 00:00:00+00:00
1           15 Jul 2009 2009-07-15 00:00:00+00:00
2              1-Feb-15 2015-02-01 00:00:00+00:00
3            12/08/2019 2019-12-08 00:00:00+00:00
4            not a date                       NaT
5  2020-08-22T00:00:00Z 2020-08-22 00:00:00+00:00
6  2021-09-11T00:00:00Z 2021-09-11 00:00:00+00:00
Date_Column                              str
Date_Column_Converted    datetime64[us, UTC]
dtype: object


In [11]:
# Create a sample DataFrame
df = pd.DataFrame({'Date': ['2023-01-01', '2023-03-31', '2023-07-15', '2024-11-20']})

# Convert to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract components
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter

print(df)

        Date  Year  Month  Quarter
0 2023-01-01  2023      1        1
1 2023-03-31  2023      3        1
2 2023-07-15  2023      7        3
3 2024-11-20  2024     11        4


In [12]:
df['YearMonth_Period'] = df['Date'].dt.to_period('M')
df['YearQuarter_Period'] = df['Date'].dt.to_period('Q')
print(df)

        Date  Year  Month  Quarter YearMonth_Period YearQuarter_Period
0 2023-01-01  2023      1        1          2023-01             2023Q1
1 2023-03-31  2023      3        1          2023-03             2023Q1
2 2023-07-15  2023      7        3          2023-07             2023Q3
3 2024-11-20  2024     11        4          2024-11             2024Q4


In [13]:
df.dtypes

Date                  datetime64[us]
Year                           int32
Month                          int32
Quarter                        int32
YearMonth_Period           period[M]
YearQuarter_Period     period[Q-DEC]
dtype: object